# Bridge — Drums `drum_humanizer.onnx` (Magenta distillation)

**Flow:** MIDI seed → **GrooVAE** teacher (`groovae_4bar`) → `distill_batch_drums` → **DrumStudentMLP** → ONNX.

Increase `n_samples` / `TRAIN_STEPS` for higher quality (longer runtime).

In [ ]:
# === 1) Dependencies ===
!pip -q install "tensorflow>=2.14,<2.17" magenta note-seq pretty_midi torch onnx onnxruntime numpy scipy matplotlib

import os, sys
REPO = "/content/Bridge"
sys.path.insert(0, os.path.join(REPO, "ml", "training") if os.path.isdir(REPO) else ".")
import numpy as np
import torch
print("Torch", torch.__version__, "CUDA", torch.cuda.is_available())

In [ ]:
# === 2) Dataset size & cache (tune for quality vs time) ===
n_samples = 500  # increase to 2000+ for smoother student
TRAIN_STEPS = 1200
BATCH = 64
LR = 1e-3
CACHE_DIR = "/content/magenta_ckpt"
# Optional: set to your MIDI path (else bundled simple_groove.mid)
USER_MIDI = None  # e.g. "/content/my_groove.mid"


In [ ]:
# === 3) Seed MIDI (fixture or user) ===
from magenta_data_utils import ensure_simple_groove_fixture, load_midi_to_note_sequence

midi_path = USER_MIDI if USER_MIDI else ensure_simple_groove_fixture()
ns = load_midi_to_note_sequence(midi_path)
print("Seed MIDI:", midi_path, "notes:", len(ns.notes))

In [ ]:
# === 4) Teacher + distillation ===
from magenta_pipeline import MagentaDrumsTeacher, distill_batch_drums

teacher = MagentaDrumsTeacher(cache_dir=CACHE_DIR)
X, Y = distill_batch_drums(teacher, n_samples=n_samples, seed=7, midi_path=midi_path)
print("X", X.shape, "Y", Y.shape)

In [ ]:
# === 5) Train student + export ONNX ===
from drum_onnx_student import train_student_from_xy, export_drum_onnx, DRUM_ONNX_IN, DRUM_ONNX_OUT

model = train_student_from_xy(X, Y, steps=TRAIN_STEPS, batch=BATCH, lr=LR)
out_dir = os.path.join(REPO, "ml", "models") if os.path.isdir(REPO) else "ml/models"
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, "drum_humanizer.onnx")
export_drum_onnx(model, out_path)
print("Saved", out_path, DRUM_ONNX_IN, "->", DRUM_ONNX_OUT, os.path.getsize(out_path), "bytes")

In [ ]:
# === 6) ONNX Runtime shape check ===
import onnxruntime as ort
sess = ort.InferenceSession(out_path, providers=["CPUExecutionProvider"])
x = np.random.rand(1, DRUM_ONNX_IN).astype(np.float32)
y = sess.run(None, {"input": x})[0]
assert y.shape == (1, DRUM_ONNX_OUT), y.shape
print("ORT OK", y.shape)

In [ ]:
# === 7) Perceptual quality (quick MIDI metrics) ===
import os, tempfile
import numpy as np
import onnxruntime as ort
from onnx_runtime_inputs import build_fixed_input
from magenta_data_utils import bridge_student_output_to_pretty_midi
from quality_metrics import perceptual_verdict, score_midi_file

INSTRUMENT = "drums"
sess = ort.InferenceSession(out_path, providers=["CPUExecutionProvider"])
_in = sess.get_inputs()[0]
knobs = np.array([0.5] * 10, dtype=np.float32)
x = build_fixed_input(INSTRUMENT, knobs).reshape(1, -1).astype(np.float32)
y = sess.run(None, {_in.name: x})[0].reshape(-1)
pm = bridge_student_output_to_pretty_midi(INSTRUMENT, y, bpm=120.0)
fd, tmp = tempfile.mkstemp(suffix=".mid"); os.close(fd)
pm.write(tmp)
metrics = score_midi_file(tmp, INSTRUMENT, bpm=120.0)
os.unlink(tmp)
print("Perceptual metrics:")
for k in ("rhythmic_consistency", "pitch_entropy", "velocity_range", "density", "groove_score", "melodic_contour_smoothness"):
    print(f"  {k}: {metrics.get(k)}")
print(perceptual_verdict(metrics, INSTRUMENT))

In [ ]:
# === 7) Download (Colab) ===
try:
    from google.colab import files
    files.download(out_path)
except ImportError:
    print("Not on Colab — ONNX at:", out_path)